# Part 2 - UWB Anchor Global Localization (GDOP & Incorrect Anchor Placement)

This notebook explores UWB-based robot localization with:

- Geometric Dilution of Precision (GDOP) analysis
- UWB anchor placement evaluation
- Anchor mis-calibration effects on pose estimation

In [1]:
# Cell 2: Imports

import numpy as np
import matplotlib.pyplot as plt

from manifpy import SE2

import sys
sys.path.insert(0, '/workspace/Python/Relative-localisation-tutorial/src')

from config import TrajectoryConfig, UWBAnchorConfig, UWBNoiseConfig
from trajectory import generate_trajectory, sgal3_to_se2_poses, add_noise
from uwb import (
    simulate_uwb_ranges,
    estimate_poses_from_ranges,
    compute_gdop_at_point,
    generate_gdop_heatmap,
    plot_anchor_positions,
    plot_gdop_heatmap
)
from plotting import plot_trajectory_2d, plot_relative_pose

ImportError: cannot import name 'plot_anchor_positions' from 'uwb' (/home/yuri/Documents/Code_base_protected/Python/Relative-localisation-tutorial/src/uwb.py)

In [ ]:
# Cell 3: Robot trajectory generation and visualization

traj_config = TrajectoryConfig(duration=50.0, timestep=0.1, seed=42)

traj_robot1 = generate_trajectory(
    duration=traj_config.duration,
    timestep=traj_config.timestep,
    seed=traj_config.seed
)

poses_robot1 = sgal3_to_se2_poses(traj_robot1.X_RRi)

fig, ax = plt.subplots(1, 1, figsize=(8, 6))
ax.plot(poses_robot1[:, 0], poses_robot1[:, 1], 'b-', label='Robot 1', alpha=0.7)

arrow_len = 0.3
step = max(1, len(poses_robot1) // 20)
for i in range(0, len(poses_robot1), step):
    x_i, y_i, theta_i = poses_robot1[i]
    dx = arrow_len * np.cos(theta_i)
    dy = arrow_len * np.sin(theta_i)
    ax.arrow(x_i, y_i, dx, dy, head_width=0.1, head_length=0.1, fc='b', ec='b')

ax.set_xlabel('x (m)')
ax.set_ylabel('y (m)')
ax.axis('equal')
ax.legend()
ax.set_title('Robot 1 Perfect Trajectory')

plt.tight_layout()
plt.show()

In [ ]:
# Cell 4: UWB anchors and noise parameters

# Define 3 anchor positions in 2D (forming a roughly equilateral triangle)
anchor_positions = np.array([
    [0.0, 0.0],      # Anchor 1 at origin
    [20.0, 0.0],     # Anchor 2 on x-axis
    [10.0, 17.32]    # Anchor 3 forming equilateral triangle (height = 20 * sqrt(3)/2)
])

# UWB noise parameters
uwb_noise_config = UWBNoiseConfig(sigma_range=0.2)  # 20 cm range noise

# Simulate UWB ranges along trajectory
ranges_robot1 = simulate_uwb_ranges(
    poses_robot1,
    anchor_positions,
    uwb_noise_config
)

# Estimate poses from noisy ranges
poses_estimated, residuals = estimate_poses_from_ranges(ranges_robot1, anchor_positions)

# Visualize: anchors + trajectory + estimated pose positions
fig, ax = plt.subplots(1, 1, figsize=(8, 6))

# Plot robot trajectory
ax.plot(poses_robot1[:, 0], poses_robot1[:, 1], 'b-', label='True trajectory', alpha=0.7)

# Plot estimated positions (dots instead of arrows for sparse points)
step = max(1, len(poses_estimated) // 50)
ax.plot(poses_estimated[::step, 0], poses_estimated[::step, 1], 'r.', 
        label='Estimated positions', markersize=4, alpha=0.8)

# Plot anchors
plot_anchor_positions(ax, anchor_positions, legend_label='UWB Anchors')

ax.set_xlabel('x (m)')
ax.set_ylabel('y (m)')
ax.axis('equal')
ax.legend()
ax.set_title('Robot Trajectory and UWB Position Estimates')

plt.tight_layout()
plt.show()

# Cell 5: GDOP Analysis

## Geometric Dilution of Precision (GDOP)

GDOP quantifies how anchor geometry affects positioning accuracy:

$$
\text{GDOP} = \sqrt{\text{trace}((J^T J)^{-1})}
$$

where $J$ is the geometry matrix of partial derivatives.

**Key principles:**
- GDOP is unitless, always ≥ 1
- Lower GDOP = better accuracy (ideal: GDOP ≈ 1)
- High GDOP occurs with poor anchor geometry (e.g., anchors in a line)
- Optimal geometry: anchors forming ~120° angles around target

In [ ]:
# Cell 6: GDOP heatmap computation and visualization

# Compute trajectory bounds for heatmap region
x_min, x_max = poses_robot1[:, 0].min() - 5, poses_robot1[:, 0].max() + 5
y_min, y_max = poses_robot1[:, 1].min() - 5, poses_robot1[:, 1].max() + 5

# Generate GDOP heatmap over workspace
X, Y, gdop_grid = generate_gdop_heatmap(
    anchor_positions,
    x_range=(x_min, x_max),
    y_range=(y_min, y_max),
    resolution=100
)

# Compute GDOP values along robot trajectory
gdop_along_trajectory = np.array([
    compute_gdop_at_point(poses_robot1[i, :2], anchor_positions)
    for i in range(len(poses_robot1))
])

# Visualize GDOP heatmap
fig, ax = plt.subplots(1, 1, figsize=(8, 6))

plot_gdop_heatmap(ax, X, Y, gdop_grid, anchor_positions=anchor_positions)

# Overlay robot trajectory on GDOP map
ax.plot(poses_robot1[:, 0], poses_robot1[:, 1], 'w-', 
        label='Robot trajectory', linewidth=2, alpha=0.8)

ax.legend()
ax.set_title('GDOP Heatmap with Robot Trajectory Overlay')

plt.tight_layout()
 plt.show()

In [ ]:
# Additional visualization: GDOP along trajectory

fig, ax = plt.subplots(1, 2, figsize=(14, 5))

t = np.arange(len(gdop_along_trajectory)) * traj_config.timestep

# GDOP over time
ax[0].plot(t, gdop_along_trajectory, 'b-', alpha=0.7)
ax[0].set_xlabel('Time (s)')
ax[0].set_ylabel('GDOP')
ax[0].set_title('GDOP Along Trajectory')
ax[0].grid(True, alpha=0.3)

# Range noise amplification estimate
est_position_error = gdop_along_trajectory * uwb_noise_config.sigma_range
ax[1].plot(t, est_position_error, 'r-', alpha=0.7)
ax[1].set_xlabel('Time (s)')
ax[1].set_ylabel('Estimated position error (m)')
ax[1].set_title('Position Error Estimate from GDOP + Range Noise')
ax[1].grid(True, alpha=0.3)

plt.tight_layout()
 plt.show()

In [ ]:
# Cell 7: Anchor mis-calibration analysis

# Define systematic offset for anchor positions (mis-calibration)
anchor_offset = np.array([1.0, 0.5])  # [delta_x, delta_y]

# "Calibrated" anchors (with mis-calibration applied)
anchor_positions_calibrated = anchor_positions + anchor_offset

# Estimate poses using mis-calibrated anchors
poses_estimated_miscal, residuals_miscal = estimate_poses_from_ranges(
    ranges_robot1,
    anchor_positions_calibrated
)

# Visualize comparison
fig, ax = plt.subplots(1, 2, figsize=(14, 6))

# Left: Position estimates comparison
ax[0].plot(poses_robot1[:, 0], poses_robot1[:, 1], 'k-', 
           label='True trajectory', alpha=0.7)
ax[0].plot(poses_estimated[:, 0], poses_estimated[:, 1], 'g-', 
           label='Estimate (perfect anchors)', alpha=0.7)
ax[0].plot(poses_estimated_miscal[:, 0], poses_estimated_miscal[:, 1], 'r--', 
           label='Estimate (mis-calibrated anchors)', alpha=0.7)

plot_anchor_positions(ax[0], anchor_positions, legend_label='True Anchors')
ax[0].scatter(anchor_positions_calibrated[:, 0], anchor_positions_calibrated[:, 1],
              c='orange', marker='x', s=150, label='Mis-calibrated anchors')

ax[0].set_xlabel('x (m)')
ax[0].set_ylabel('y (m)')
ax[0].axis('equal')
ax[0].legend()
ax[0].set_title('Position Estimates: Perfect vs Mis-calibrated Anchors')

# Right: Error accumulation over time
t = np.arange(len(poses_robot1)) * traj_config.timestep
pos_error_perfect = np.sqrt((poses_estimated[:, 0] - poses_robot1[:, 0])**2 + 
                            (poses_estimated[:, 1] - poses_robot1[:, 1])**2)
pos_error_miscal = np.sqrt((poses_estimated_miscal[:, 0] - poses_robot1[:, 0])**2 +
                           (poses_estimated_miscal[:, 1] - poses_robot1[:, 1])**2)

ax[1].plot(t, pos_error_perfect, 'g-', label='Perfect anchors', alpha=0.7)
ax[1].plot(t, pos_error_miscal, 'r--', label='Mis-calibrated anchors', alpha=0.7)

ax[1].set_xlabel('Time (s)')
ax[1].set_ylabel('Position error (m)')
ax[1].legend()
ax[1].set_title('Position Estimation Error Over Time')
ax[1].grid(True, alpha=0.3)

plt.tight_layout()
 plt.show()

In [ ]:
# Summary statistics

print("=== UWB Localization Performance ===")
print(f"Perfect anchors - Final position error: {pos_error_perfect[-1]:.3f} m")
print(f"Perfect anchors - Mean position error: {np.mean(pos_error_perfect):.3f} m")
print()
print(f"Mis-calibrated anchors - Final position error: {pos_error_miscal[-1]:.3f} m")
print(f"Mis-calibrated anchors - Mean position error: {np.mean(pos_error_miscal):.3f} m")
print()
print(f"Anchor calibration bias magnitude: {np.linalg.norm(anchor_offset):.2f} m")

# Cell 8: Conclusion

## Summary of Key Findings

### 1. UWB Trilateration Performance
- From range measurements to 3 anchors, we estimated robot positions via trilateration
- Position estimates show small errors due to Gaussian range noise

### 2. GDOP Analysis
- GDOP quantifies geometry-dependent error amplification
- Lower GDOP values indicate better anchor geometry
- Heatmap shows regions where positioning accuracy is optimal/poor
- Trajectory should avoid high-GDOP regions for best accuracy

### 3. Anchor Mis-calibration Effects
- Systematic bias in anchor positions directly translates to position estimation error
- Error grows over time as the robot moves away from anchors
- Mis-calibration can dominate performance compared to range measurement noise

## Connection to Part 1

In Part 1, we saw that global pose noise propagates into relative pose estimates.

Here, we see a similar but more severe effect:
- Global pose uncertainty comes from both measurement noise (GDOP) and anchor calibration errors
- Position estimation error can be much larger than in Part 1 due to GDOP amplification
- Anchor mis-calibration creates systematic biases that accumulate across the trajectory

## Recommendations for UWB Deployment

1. **Anchor geometry**: Place anchors at ~120° angles around operating area
2. **GDOP mapping**: Use GDOP heatmaps to identify high-error regions before deployment
3. **Calibration**: Regularly calibrate anchor positions (e.g., using total station or precise RTK-GNSS)
4. **Redundancy**: Add 4th+ anchors for outlier rejection and improved geometry